# Architecture Design for Maze Solver

## 1. Analysis of Lab3IA
- **Reusable:** `TreeNode` structure (adapted for coordinates), Queue structures (`FIFOQueue`, `LIFOQueue`, `PriorityQueue` which can be adapted slightly).
- **Adapted:** Graph representation changes to a Grid representation (matrix). We need coordinate-based neighbor generation instead of a predefined edge list (`grafo`). The general search loop structure can be adapted.
- **Implemented from scratch:** Maze loading from `.txt`, Start/Goal location in a matrix, Grid-based valid neighbor generation respecting priority (Up, Right, Down, Left), Heuristic functions strictly for 2D grids (Manhattan, Euclidean), random valid start position generation, metrics tracking (branching factor calculation, timing).

## 2. Project Architecture

The notebook will be structured logically into the following sections:

1. **Imports & Initial Setup**: Required standard libraries (`time`, `math`, `random`).
2. **Maze Loading & Utilities**: Functions to read the `.txt` file, convert to a 2D array, and find Start (`2`) and Goal (`3`).
3. **Core Data Structures**: 
   - Node (`TreeNode`) adapted with `(x, y)` state.
   - Priority and standard queues tailored for the algorithms.
4. **Grid Mechanics**: 
   - Define movement directions (Up, Right, Down, Left).
   - Generate valid neighbors (check bounds and walls `1`).
5. **Heuristics**:
   - Manhattan and Euclidean distance functions.
6. **Search Algorithms Base**:
   - A modular search engine that accepts different queuing strategies and heuristics to execute BFS, DFS, Greedy, and A*.
7. **Metrics Collection**:
   - Wrapper or integrated counters for execution time, max memory (or nodes expanded), path length, and branching factor.
8. **Experiments**:
   - Scenario 1: Predefined Start to Goal.
   - Scenario 2: Random Start to Goal.
9. **Results & Comparison**: tabulated outputs.

---
Let's begin with **Step 1: Load the maze** into a matrix natively.

In [1]:
# STEP 1: Load the maze

# load the text file and convert to integer matrix
def load_maze(file_path):
    maze = []
    # read file lines and parse integers
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            row = [int(char) for char in line.strip() if char.isdigit()]
            if row:
                maze.append(row)
    return maze

# verify loading (assuming test_maze.txt exists)
# maze_grid = load_maze('test_maze.txt')
# print(f"Maze dimensions: {len(maze_grid)}x{len(maze_grid[0]) if maze_grid else 0}")


In [2]:
# STEP 2, 3 & 4: Mechanics and grid interpretation

# find specific value in grid
def find_position(maze, value):
    for r, row in enumerate(maze):
        for c, val in enumerate(row):
            if val == value:
                return (r, c)
    return None

# define valid moves with UP, RIGHT, DOWN, LEFT priority
DIRECTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

# generate valid neighbors
def get_neighbors(maze, position):
    r, c = position
    neighbors = []
    rows = len(maze)
    cols = len(maze[0]) if rows > 0 else 0
    
    for dr, dc in DIRECTIONS:
        nr, nc = r + dr, c + dc
        # check bounds and if it's not a wall
        if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] != 1:
            neighbors.append((nr, nc))
            
    return neighbors


In [3]:
# STEP 5: Reusing and adapting the tree node for pathfinding

class TreeNode:
    def __init__(self, position, path_cost=0, heuristic=0, parent=None):
        self.position = position  # (row, col) tuple
        self.path_cost = path_cost  # g(n)
        self.heuristic = heuristic  # h(n)
        self.parent = parent  # allows path reconstruction
        
    def total_cost(self):
        # f(n) = g(n) + h(n)
        return self.path_cost + self.heuristic

    def __lt__(self, other):
        # tie breaker for priority queues (not strictly defined, but useful)
        return self.total_cost() < other.total_cost()

    def __repr__(self):
        return f"Node({self.position}, g: {self.path_cost}, h: {self.heuristic})"

# extract path to goal
def reconstruct_path(node):
    path = []
    current = node
    while current:
        path.append(current.position)
        current = current.parent
    return path[::-1]  # reverse to start->goal

---

**Check-in!**
We have successfully implemented Steps 1 through 5. The project architecture relies on reusing standard Queue/Node approaches while adapting matrix mechanics. 
Please review and confirm to proceed with Step 6 (BFS)!

In [4]:
# STEP 6: Implement BFS

import time

# standard queue for BFS
class FIFOQueue:
    def __init__(self):
        self.items = []
        
    def is_empty(self):
        return len(self.items) == 0
        
    def push(self, item):
        self.items.append(item)
        
    def pop(self):
        return self.items.pop(0) if not self.is_empty() else None

# basic BFS algorithm
def bfs(maze, start_pos, goal_pos):
    # setup starting node
    start_node = TreeNode(start_pos)
    frontier = FIFOQueue()
    frontier.push(start_node)
    
    # initialize metrics
    explored_nodes = 0
    visited = set()
    start_time = time.time()
    
    while not frontier.is_empty():
        current_node = frontier.pop()
        explored_nodes += 1
        
        # goal check
        if current_node.position == goal_pos:
            exec_time = time.time() - start_time
            path = reconstruct_path(current_node)
            return path, explored_nodes, exec_time
            
        visited.add(current_node.position)
        
        # expand neighbors
        for neighbor_pos in get_neighbors(maze, current_node.position):
            if neighbor_pos not in visited:
                # avoid duplicate pushes
                visited.add(neighbor_pos)
                child = TreeNode(neighbor_pos, path_cost=current_node.path_cost + 1, parent=current_node)
                frontier.push(child)
                
    exec_time = time.time() - start_time
    return None, explored_nodes, exec_time


In [5]:
# STEP 7: Implement DFS

# standard LIFO stack for DFS
class LIFOQueue:
    def __init__(self):
        self.items = []
        
    def is_empty(self):
        return len(self.items) == 0
        
    def push(self, item):
        self.items.append(item)
        
    def pop(self):
        return self.items.pop() if not self.is_empty() else None

# basic DFS algorithm
def dfs(maze, start_pos, goal_pos):
    # setup starting node
    start_node = TreeNode(start_pos)
    frontier = LIFOQueue()
    frontier.push(start_node)
    
    # initialize metrics
    explored_nodes = 0
    visited = set()
    start_time = time.time()
    
    while not frontier.is_empty():
        current_node = frontier.pop()
        explored_nodes += 1
        
        # goal check
        if current_node.position == goal_pos:
            exec_time = time.time() - start_time
            path = reconstruct_path(current_node)
            return path, explored_nodes, exec_time
            
        visited.add(current_node.position)
        
        # expand neighbors
        # reversing neighbors to maintain Up, Right, Down, Left priority when popping
        neighbors = get_neighbors(maze, current_node.position)
        for neighbor_pos in reversed(neighbors):
            if neighbor_pos not in visited:
                child = TreeNode(neighbor_pos, path_cost=current_node.path_cost + 1, parent=current_node)
                frontier.push(child)
                
    exec_time = time.time() - start_time
    return None, explored_nodes, exec_time


In [6]:
# STEP 8: Heuristics

import math

# calculate manhattan distance (|x1 - x2| + |y1 - y2|)
def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

# calculate straight-line euclidean distance
def euclidean_distance(pos1, pos2):
    return math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
